In [ ]:
import numpy as np 

import chemfiles
from atomcorr.analysis.onsager_coefficients import OnsagerCoefficients
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LogNorm
import mdsuite as mds
from mdsuite.utils import Units

import h5py as hf

In [ ]:
project = mds.Project("water-lammps")

prefix = "/work/stovey/lj_nvt.lammpstraj"

water_chemical = project.add_experiment(
    name="lj",
    timestep=0.001,
    temperature=84.0,
    units="real",
    simulation_data = prefix
)

In [ ]:
with hf.File("water-lammps/lj/database.hdf5") as db:

    velocity = np.concatenate(
        (
            np.transpose(db['1']["Velocities"][:], (1, 0, 2)), 
            np.transpose(db['2']["Velocities"][:], (1, 0, 2))
        ), 
        axis=1
    )

In [ ]:
def compute_atom_correlation(data):
    """
    Compute atom-wise correlation tensor.
    """    
    calculator = OnsagerCoefficients()

    correlation_matrix = calculator._compute_correlation_matrix(
        data,
        data,
        correlation_time=500,
        data_range=500
    )

    return correlation_matrix


In [ ]:
def normalize_covariance_matrix(covariance_matrix: np.ndarray):
    """
    Method for normalizing a covariance matrix.
    Returns
    -------
    normalized_covariance_matrix : np.ndarray
            A normalized covariance matrix, i.e, the matrix given if all of its inputs
            had been normalized.
    """
    order = np.shape(covariance_matrix)[0]

    diagonals = np.diagonal(covariance_matrix)

    repeated_diagonals = np.repeat(diagonals[None, :], order, axis=0)

    normalizing_matrix = np.sqrt(repeated_diagonals * repeated_diagonals.T)

    return covariance_matrix / normalizing_matrix



In [ ]:
matrix = compute_atom_correlation(velocity)

In [ ]:
np.save("matrix.npy", matrix)

In [ ]:
sns.heatmap(
    normalize_covariance_matrix(matrix), 
    robust=True,
    cmap='flare',
    yticklabels=False,
    xticklabels=False,
    # norm=LogNorm()
)
plt.show()

In [ ]:
sns.histplot(np.concatenate(matrix), stat="probability")

In [ ]:
eigs = np.linalg.eigh(matrix)[0]

In [ ]:
plt.plot(eigs[::-1])
# plt.yscale("log")

In [ ]:
sns.kdeplot(eigs)

In [ ]:
svd = np.linalg.svd(matrix)

In [ ]:
plt.plot(svd[1])

In [ ]:
sns.kdeplot(svd[1])